In [1]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam import AblationCAM
from pytorch_grad_cam.ablation_layer import AblationLayerVit
from pytorch_grad_cam import EigenGradCAM
from pytorch_grad_cam import XGradCAM
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.model_selection import train_test_split
from einops import rearrange
from einops.layers.torch import Rearrange
import pandas as pd
import random
import gc

In [2]:
def pair(t):
    return t if isinstance(t, tuple) else (t, t)

def posemb_sincos_2d(h, w, dim, temperature: int = 10000, dtype = torch.float32):
    y, x = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
    assert (dim % 4) == 0, "feature dimension must be multiple of 4 for sincos emb"
    omega = torch.arange(dim // 4) / (dim // 4 - 1)
    omega = 1.0 / (temperature ** omega)

    y = y.flatten()[:, None] * omega[None, :]
    x = x.flatten()[:, None] * omega[None, :]
    pe = torch.cat((x.sin(), x.cos(), y.sin(), y.cos()), dim=1)
    return pe.type(dtype)

# classes

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64):
        super().__init__()
        inner_dim = dim_head *  heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.norm = nn.LayerNorm(dim)

        self.attend = nn.Softmax(dim = -1)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)
        self.to_out = nn.Linear(inner_dim, dim, bias = False)

    def forward(self, x):
        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, heads = heads, dim_head = dim_head),
                FeedForward(dim, mlp_dim)
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return self.norm(x)

class SimpleViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, channels = 3, dim_head = 64):
        super().__init__()
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        patch_dim = channels * patch_height * patch_width

        self.to_patch_embedding = nn.Sequential(
            Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1 = patch_height, p2 = patch_width),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )

        self.pos_embedding = posemb_sincos_2d(
            h = image_height // patch_height,
            w = image_width // patch_width,
            dim = dim,
        ) 

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim)
        self.linear_head = nn.Linear(dim, num_classes)

    def forward_features(self, img):
        device = img.device

        x = self.to_patch_embedding(img)
        x += self.pos_embedding.to(device, dtype=x.dtype)

        x = self.transformer(x)
        x = x.mean(dim=1)
        return x

    def forward(self, img):
        x = self.forward_features(img)
        logits = self.linear_head(x)
        return logits

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Treino sem sampler e sem augmentation e sem segmentação

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\SimpleViT_4_class_SWeightSSeg.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=256, bias=True)
    (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=256, out_features=756, bias=False)
          (to_out): Linear(in_features=252, out_features=256, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=256, out_features=1024, bias=True)
            (2): GELU(approximate='none')
          

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(19)

mean = torch.tensor([0.7261, 0.7059, 0.7086])
std  = torch.tensor([0.1725, 0.1932, 0.1980])
normalizador = transforms.Normalize(mean=mean, std=std)

## Treino com sampler e com augmentation e sem segmentação

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\SimpleViT_4_class_CWeightSSeg.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=256, bias=True)
    (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=256, out_features=768, bias=False)
          (to_out): Linear(in_features=256, out_features=256, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=256, out_features=1024, bias=True)
            (2): GELU(approximate='none')
          

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(26)

mean = torch.tensor([0.7261, 0.7059, 0.7086])
std  = torch.tensor([0.1725, 0.1932, 0.1980])
normalizador = transforms.Normalize(mean=mean, std=std)

## Treino com sampler e augmentation e segmentação (fundo preto)

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\SimpleViT_4_class_CWeightCSegPreto.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=128, bias=True)
    (3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-3): 4 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=128, out_features=384, bias=False)
          (to_out): Linear(in_features=128, out_features=128, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=128, out_features=1024, bias=True)
            (2): GELU(approximate='none')
          

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(7)

mean = torch.tensor([0.1464, 0.1327, 0.1299])
std  = torch.tensor([0.2578, 0.2398, 0.2336])
normalizador = transforms.Normalize(mean=mean, std=std)

## Treino com sampler e augmentation e com segmentação (fundo branco)

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\SimpleViT_4_class_CWeightCSegBranco.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=256, bias=True)
    (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-6): 7 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=256, out_features=768, bias=False)
          (to_out): Linear(in_features=256, out_features=256, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=256, out_features=384, bias=True)
            (2): GELU(approximate='none')
           

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(15)

mean = torch.tensor([0.8731, 0.8594, 0.8566])
std  = torch.tensor([0.2293, 0.2517, 0.2539])
normalizador = transforms.Normalize(mean=mean, std=std)

## Treino com sampler e com augmentation e com segmentação (fundo de ruído gaussiano)

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\SimpleViT_4_class_CWeightCSegApenasRuido.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=512, bias=True)
    (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=512, out_features=1536, bias=False)
          (to_out): Linear(in_features=512, out_features=512, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=512, out_features=1024, bias=True)
            (2): GELU(approximate='none')
         

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(4)

mean = torch.tensor([0.4962, 0.4825, 0.4797])
std  = torch.tensor([0.2620, 0.2633, 0.2619])
normalizador = transforms.Normalize(mean=mean, std=std)

## Treino com sampler e com augmentation e com segmentação (misto: imagens normais e imagens com fundo de ruído gaussiano)

In [4]:
CAMINHO_PESOS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\SimpleViT_4_class_CWeightCSegMisto.pt"
CAMINHO_SALVAR = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI'
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

CLASSES_DICIONARIO = {0: "Míldio", 1: "Oídio", 2: "Phakopsora", 3: "Puccinias"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
checkpoint = torch.load(CAMINHO_PESOS,
                        map_location=device,
                        weights_only=True)

config = checkpoint["config"]

model = SimpleViT(
    image_size=64,
    patch_size=config["patch_size"],
    num_classes=4,
    dim=config["dim"],
    depth=config["depth"],
    heads=config["heads"],
    mlp_dim=config["mlp_dim"],
    dim_head=config["dim_head"]
).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

C:\Users\Fabricio\anaconda3\envs\yolov8\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


SimpleViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=4, p2=4)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=48, out_features=192, bias=True)
    (3): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Transformer(
    (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (to_qkv): Linear(in_features=192, out_features=576, bias=False)
          (to_out): Linear(in_features=192, out_features=192, bias=False)
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=192, out_features=384, bias=True)
            (2): GELU(approximate='none')
           

In [6]:
def obter_reshape_transform(patch_size):
    def reshape_transform(tensor): return tensor.reshape(tensor.size(0), 64//patch_size, 64//patch_size, tensor.size(2)).transpose(2, 3).transpose(1, 2)
    return reshape_transform

target_layers = [model.transformer.layers[-1][0].to_out]
reshape_func = obter_reshape_transform(config["patch_size"])

set_seed(18)

mean = torch.tensor([0.6111, 0.5941, 0.5941])
std  = torch.tensor([0.2493, 0.2559, 0.2584])
normalizador = transforms.Normalize(mean=mean, std=std)

## O restante do pipeline se mantém igual

In [7]:
test_filesBlack = np.load(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\FilesLabels_SFP\test_files_SFP.npy', allow_pickle=True).tolist()
test_labels = np.load(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\FilesLabels_SFP\test_labels_SFP.npy', allow_pickle=True).tolist()
test_filesClean = np.load(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\FilesLabels_Filtrado\test_files_Filtrado.npy', allow_pickle=True).tolist()

In [8]:
def processar_imagem(caminho):
    
    img_bgr = cv2.imread(caminho)
    
    img_rgb_float = cv2.cvtColor(cv2.resize(img_bgr, (64, 64)), cv2.COLOR_BGR2RGB) / 255.0
    
    tensor_img = torch.tensor(img_rgb_float, dtype=torch.float32).permute(2, 0, 1)
    input_tensor = normalizador(tensor_img).unsqueeze(0).to(device)
    
    return input_tensor, img_rgb_float

In [9]:
def gerar_imagemNoise(caminho, idx):
    
    img_seg = cv2.imread(caminho)
    img_seg = cv2.cvtColor(img_seg, cv2.COLOR_BGR2RGB)

    img_float = img_seg.astype(np.float32) / 255.0

    mascara = (img_float.max(axis=2, keepdims=True) > 0).astype(np.float32)

    rng_local = np.random.RandomState(idx)
            
    r = rng_local.rand()

    if r < 0.33:
        ruido = rng_local.normal(0.5, 0.2, img_float.shape)
    elif r < 0.66:
        ruido = rng_local.uniform(0, 1, img_float.shape)
    else:
        ruido = rng_local.normal(0.5, 0.4, img_float.shape)

    ruido = np.clip(ruido, 0, 1).astype(np.float32)

    img = (img_float * mascara + ruido * (1 - mascara)) * 255
    img = img.astype(np.uint8)

    return img, mascara

In [10]:
def processar_imagemNoise(caminho, idx):
    img_rgb, mascara = gerar_imagemNoise(caminho, idx)

    img_rgb_float = cv2.resize(img_rgb, (64, 64)) / 255.0

    tensor_img = torch.tensor(img_rgb_float, dtype=torch.float32).permute(2, 0, 1)
    input_tensor = normalizador(tensor_img).unsqueeze(0).to(device)
    
    mascara = cv2.resize(mascara.squeeze(), (64, 64), interpolation=cv2.INTER_NEAREST)

    mascara = mascara.astype(np.float32)

    return input_tensor, img_rgb_float, mascara

In [11]:
def gerar_heatmaps(model, input_tensor, cam, img_rgb_float, CLASSES_DICIONARIO, label_real, nome_metodo):

    input_tensor = input_tensor.requires_grad_(True)
    saida = model(input_tensor)
    probs = torch.nn.functional.softmax(saida, dim=1)
    classe_predita = torch.argmax(saida, dim=1).item()
    confianca = probs[0, classe_predita].item() * 100
    confianca_real = probs[0, label_real].item() * 100

    nome_real = CLASSES_DICIONARIO[label_real]
    nome_pred = CLASSES_DICIONARIO[classe_predita]

    predicao_correta = (label_real == classe_predita)

    cor_titulo = 'darkgreen' if predicao_correta else 'darkred'
    
    metodos_com_eigen_smooth = {
        "GradCAM",
        "AblationCAM",
        "XGradCAM",
        "EigenGradCAM"
    }

    usar_eigen_smooth = nome_metodo in metodos_com_eigen_smooth

    if predicao_correta:
        
        cam_kwargs = {
            "input_tensor": input_tensor,
            "targets": [ClassifierOutputTarget(classe_predita)]
        }

        if usar_eigen_smooth:
            cam_kwargs["eigen_smooth"] = True
        
        grayscale_cam = cam(**cam_kwargs)[0, :]
            
        grayscale_cam = np.maximum(grayscale_cam, 0)
        
        return {
            "predicao_correta": True,
            "cam_pred": grayscale_cam,
            "cam_real": None,
            "classe_predita": classe_predita,
            "confianca": confianca,
            "confianca_real": confianca_real,
            "nome_real": nome_real,
            "nome_pred": nome_pred,
            "usar_eigen_smooth": usar_eigen_smooth
        }

    # PREDIÇÃO ERRADA

    else:

        cam_kwargs_pred = {
            "input_tensor": input_tensor,
            "targets": [ClassifierOutputTarget(classe_predita)]
        }
        
        cam_kwargs_real = {
            "input_tensor": input_tensor,
            "targets": [ClassifierOutputTarget(label_real)]
        }

        if usar_eigen_smooth:
            cam_kwargs_pred["eigen_smooth"] = True
            cam_kwargs_real["eigen_smooth"] = True

        grayscale_cam_pred = cam(**cam_kwargs_pred)[0, :]

        grayscale_cam_real = cam(**cam_kwargs_real)[0, :]
            
        grayscale_cam_pred = np.maximum(grayscale_cam_pred, 0)
        grayscale_cam_real = np.maximum(grayscale_cam_real, 0)
            
        return {
            "predicao_correta": False,
            "cam_pred": grayscale_cam_pred,
            "cam_real": grayscale_cam_real,
            "classe_predita": classe_predita,
            "confianca": confianca,
            "confianca_real": confianca_real,
            "nome_real": nome_real,
            "nome_pred": nome_pred,
            "usar_eigen_smooth": usar_eigen_smooth
        }    

In [12]:
def plotar_cam(grayscale_cam, img_rgb_float, nome_arquivo_original, nome_metodo, CAMINHO_METODO):
    
    cor_titulo = ("darkgreen" if grayscale_cam["predicao_correta"] else "darkred")
    
    if grayscale_cam["predicao_correta"]:
        
        cam_pred = grayscale_cam["cam_pred"]
        
        visualization = show_cam_on_image(img_rgb_float, cam_pred, use_rgb=True)

        fig, axs = plt.subplots(1, 2, figsize=(8, 4))

        fig.suptitle(f"Real: {grayscale_cam['nome_real']} | IA: {grayscale_cam['nome_pred']} ({grayscale_cam['confianca']:.1f}%)", fontsize=14, fontweight='bold', color=cor_titulo)

        axs[0].imshow(img_rgb_float)
        axs[0].set_title(f"Arquivo: {nome_arquivo_original}", fontsize=10)
        axs[0].axis('off')

        axs[1].imshow(visualization)
        axs[1].set_title(f"{nome_metodo} - Predita", fontsize=12)
        axs[1].axis('off')
    
    else:
        
        cam_pred = grayscale_cam["cam_pred"]
        # Heatmap da classe predita
        visualization_pred = show_cam_on_image(img_rgb_float, cam_pred, use_rgb=True)
        
        # Heatmap da classe real
        cam_real = grayscale_cam["cam_real"]
        visualization_real = show_cam_on_image(img_rgb_float, cam_real, use_rgb=True)

        fig, axs = plt.subplots(1, 3, figsize=(12, 4))

        fig.suptitle(f"Real: {grayscale_cam['nome_real']} | IA: {grayscale_cam['nome_pred']} ({grayscale_cam['confianca']:.1f}%)",
                     fontsize=14, fontweight='bold', color=cor_titulo)

        axs[0].imshow(img_rgb_float)
        axs[0].set_title(f"Arquivo: {nome_arquivo_original}", fontsize=10)
        axs[0].axis('off')

        axs[1].imshow(visualization_pred)
        axs[1].set_title(f"{nome_metodo} - Predita ({grayscale_cam['confianca']:.1f}%)")
        axs[1].axis('off')

        axs[2].imshow(visualization_real)
        axs[2].set_title(f"{nome_metodo} - Classe Real ({grayscale_cam['confianca_real']:.1f}%)")
        axs[2].axis('off')
        
    nome_salvar = f"[{grayscale_cam['nome_real']}]_pred_{grayscale_cam['nome_pred']}_{nome_arquivo_original}"
    caminho_salvar = os.path.join(CAMINHO_METODO, f"{nome_salvar}.jpg")

    plt.tight_layout()
    plt.savefig(caminho_salvar, dpi=150)
    plt.close(fig)


In [12]:
def energy_inside(grayscale_cam, mascara):
    
    cam = (grayscale_cam["cam_real"] if grayscale_cam["cam_real"] is not None else grayscale_cam["cam_pred"])

    total_energy = np.sum(cam)

    if total_energy > 0:
        score = np.sum(cam * mascara) / total_energy
    else:
        score = 0.0

    return score

In [13]:
def calcular_iou(grayscale_cam, mascara, percentile=90):

    cam = (grayscale_cam["cam_real"] if grayscale_cam["cam_real"] is not None else grayscale_cam["cam_pred"])
    
    threshold = np.percentile(cam, percentile)

    cam_bin = cam >= threshold

    lesion_bin = mascara.squeeze() > 0

    intersection = np.logical_and(cam_bin, lesion_bin).sum()

    union = np.logical_or(cam_bin, lesion_bin).sum()

    return intersection / union if union > 0 else 0

In [14]:
# AblationCAM
cam = AblationCAM(
                model=model, target_layers=target_layers, 
                reshape_transform=reshape_func,
                ablation_layer=AblationLayerVit(), batch_size=64
)

nome_metodo = "AblationCAM"

In [14]:
# GradCAM
cam = GradCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "GradCAM"

In [14]:
# EigenCAM
cam = EigenCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "EigenCAM"

In [14]:
# EigenGradCAM
cam = EigenGradCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "EigenGradCAM"

In [14]:
# XGradCAM
cam = XGradCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "XGradCAM"

In [14]:
# HiResCAM
cam = HiResCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "HiResCAM"

In [14]:
# LayerCAM
cam = LayerCAM(
                model=model, target_layers=target_layers,
                reshape_transform=reshape_func
)

nome_metodo = "LayerCAM"

In [15]:
resultados = []

In [16]:
#LIMITE_POR_CLASSE = 50

'''
CAMINHO_SALVAR_CLEAN = os.path.join(CAMINHO_SALVAR, "CleanFull")
os.makedirs(CAMINHO_SALVAR_CLEAN, exist_ok=True)

CAMINHO_SALVAR_NOISE = os.path.join(CAMINHO_SALVAR, "NoiseFull")
os.makedirs(CAMINHO_SALVAR_NOISE, exist_ok=True)
'''

CAMINHO_SALVAR = os.path.join(CAMINHO_SALVAR, "FullTestSet")
os.makedirs(CAMINHO_SALVAR, exist_ok=True)

contagem_classes = {0: 0, 1: 0,  2: 0, 3: 0}

'''
CAMINHO_METODO_CLEAN = os.path.join(CAMINHO_SALVAR_CLEAN, nome_metodo)
os.makedirs(CAMINHO_METODO_CLEAN, exist_ok=True)

CAMINHO_METODO_NOISE = os.path.join(CAMINHO_SALVAR_NOISE, nome_metodo)
os.makedirs(CAMINHO_METODO_NOISE, exist_ok=True)
'''

print(f"\n{nome_metodo}")
    
for idx, label_real in enumerate(test_labels):
    '''
    if contagem_classes[label_real] >= LIMITE_POR_CLASSE:
        if all(c >= LIMITE_POR_CLASSE for c in contagem_classes.values()):
            break
        continue
    '''
    
    nome_arquivo = os.path.basename(test_filesClean[idx])
    
    # Noise
    input_tensorNoise, img_rgb_floatNoise, mascara = processar_imagemNoise(test_filesBlack[idx], idx)
    
    grayscale_camNoise = gerar_heatmaps(model, input_tensorNoise, cam, img_rgb_floatNoise, CLASSES_DICIONARIO, label_real, nome_metodo)
    # plotar_cam(grayscale_camNoise, img_rgb_floatNoise, nome_arquivo, nome_metodo, CAMINHO_METODO_NOISE)
    scoreNoise = energy_inside(grayscale_camNoise, mascara)
    iouNoise = calcular_iou(grayscale_camNoise, mascara)
    
    resultados.append({
        "arquivo": nome_arquivo,
        "classe_real": label_real,
        "classe_predita": grayscale_camNoise["classe_predita"],
        "predicao_correta": grayscale_camNoise["predicao_correta"],
        "confianca_predita": grayscale_camNoise["confianca"],
        "energy_inside": scoreNoise,
        "iou": iouNoise,
        "metodo": nome_metodo,
        "eigen_smooth": grayscale_camNoise["usar_eigen_smooth"],
        "condicao": "Noise"
    })
    
    # Clean
    input_tensorClean, img_rgb_floatClean = processar_imagem(test_filesClean[idx])
    
    grayscale_camClean = gerar_heatmaps(model, input_tensorClean, cam, img_rgb_floatClean, CLASSES_DICIONARIO, label_real, nome_metodo)
    # plotar_cam(grayscale_camClean, img_rgb_floatClean, nome_arquivo, nome_metodo, CAMINHO_METODO_CLEAN)
    scoreClean = energy_inside(grayscale_camClean, mascara)
    iouClean = calcular_iou(grayscale_camClean, mascara)
    
    resultados.append({
        "arquivo": nome_arquivo,
        "classe_real": label_real,
        "classe_predita": grayscale_camClean["classe_predita"],
        "predicao_correta": grayscale_camClean["predicao_correta"],
        "confianca_predita": grayscale_camClean["confianca"],
        "energy_inside": scoreClean,
        "iou": iouClean,
        "metodo": nome_metodo,
        "eigen_smooth": grayscale_camClean["usar_eigen_smooth"],
        "condicao": "Clean"
    })
    
    contagem_classes[label_real] += 1
       
    if sum(contagem_classes.values()) % 50 == 0:
        print(f"Progresso atual: Míldio {contagem_classes[0]} | Oídio {contagem_classes[1]} | Ferrugem {contagem_classes[2]} | Puccinias {contagem_classes[3]}")
        
    # Limpar a memória
    del grayscale_camNoise
    del grayscale_camClean
    del input_tensorNoise
    del input_tensorClean
    del img_rgb_floatNoise
    del img_rgb_floatClean
    
    if (idx + 1) % 500 == 0:
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


LayerCAM
Progresso atual: Míldio 3 | Oídio 6 | Ferrugem 38 | Puccinias 3
Progresso atual: Míldio 5 | Oídio 9 | Ferrugem 77 | Puccinias 9
Progresso atual: Míldio 7 | Oídio 12 | Ferrugem 117 | Puccinias 14
Progresso atual: Míldio 10 | Oídio 18 | Ferrugem 154 | Puccinias 18
Progresso atual: Míldio 11 | Oídio 24 | Ferrugem 194 | Puccinias 21
Progresso atual: Míldio 13 | Oídio 29 | Ferrugem 233 | Puccinias 25
Progresso atual: Míldio 15 | Oídio 35 | Ferrugem 271 | Puccinias 29
Progresso atual: Míldio 20 | Oídio 42 | Ferrugem 301 | Puccinias 37
Progresso atual: Míldio 22 | Oídio 46 | Ferrugem 342 | Puccinias 40
Progresso atual: Míldio 23 | Oídio 54 | Ferrugem 380 | Puccinias 43
Progresso atual: Míldio 26 | Oídio 61 | Ferrugem 418 | Puccinias 45
Progresso atual: Míldio 28 | Oídio 67 | Ferrugem 453 | Puccinias 52
Progresso atual: Míldio 31 | Oídio 72 | Ferrugem 491 | Puccinias 56
Progresso atual: Míldio 32 | Oídio 74 | Ferrugem 535 | Puccinias 59
Progresso atual: Míldio 33 | Oídio 82 | Ferruge

Progresso atual: Míldio 254 | Oídio 669 | Ferrugem 4468 | Puccinias 459
Progresso atual: Míldio 255 | Oídio 672 | Ferrugem 4507 | Puccinias 466
Progresso atual: Míldio 255 | Oídio 682 | Ferrugem 4542 | Puccinias 471
Progresso atual: Míldio 257 | Oídio 686 | Ferrugem 4580 | Puccinias 477
Progresso atual: Míldio 258 | Oídio 688 | Ferrugem 4621 | Puccinias 483
Progresso atual: Míldio 261 | Oídio 698 | Ferrugem 4655 | Puccinias 486
Progresso atual: Míldio 266 | Oídio 701 | Ferrugem 4694 | Puccinias 489
Progresso atual: Míldio 268 | Oídio 706 | Ferrugem 4732 | Puccinias 494
Progresso atual: Míldio 270 | Oídio 716 | Ferrugem 4769 | Puccinias 495
Progresso atual: Míldio 274 | Oídio 721 | Ferrugem 4804 | Puccinias 501
Progresso atual: Míldio 279 | Oídio 725 | Ferrugem 4844 | Puccinias 502
Progresso atual: Míldio 281 | Oídio 734 | Ferrugem 4881 | Puccinias 504
Progresso atual: Míldio 281 | Oídio 739 | Ferrugem 4919 | Puccinias 511
Progresso atual: Míldio 283 | Oídio 746 | Ferrugem 4958 | Puccin

In [31]:
df_Clean = pd.DataFrame(resultadosClean)

df_Clean.to_csv(
    os.path.join(
        CAMINHO_SALVAR_CLEAN,
        "metricas_Clean.csv"
    ),
    index=False
)

df_Noise = pd.DataFrame(resultadosNoise)

df_Noise.to_csv(
    os.path.join(
        CAMINHO_SALVAR_NOISE,
        "metricas_Noise.csv"
    ),
    index=False
)


In [17]:
df = pd.DataFrame(resultados)

df.to_csv(
    os.path.join(
        CAMINHO_SALVAR,
        f"metricasFull_{nome_metodo}.csv"
    ),
    index=False
)